# Data Acquisition: On-Chain & Market Data

This notebook handles the ingestion of quantitative market data and on-chain metrics from public endpoints.

**Data Sources:**
1. Historical OHLCV price data (CoinGecko)
2. Live order book data (Kraken via CCXT)
3. On-chain Ethereum metrics (Public RPC)
4. DeFi liquidity pool data (Uniswap V3 via The Graph)

Data is persisted to `/data/raw/` for downstream processing.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import sys
sys.path.append('/content/drive/MyDrive/crypto_quant_project/utils')
import config

%pip install -q ccxt pycoingecko web3 pandas numpy requests

import pandas as pd
import numpy as np
import requests
import time
import os
from datetime import datetime, timedelta
from pycoingecko import CoinGeckoAPI
import ccxt
from web3 import Web3

print('✅ Setup complete')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Config loaded
✅ Setup complete


## 1. Historical Price Data Acquisition

In [ ]:
cg = CoinGeckoAPI()

def fetch_ohlc(coin_id: str, days: int = 365) -> pd.DataFrame:
    """Fetches historical OHLC data for a given asset."""
    print(f'Fetching {days} days of OHLC for {coin_id}...')
    raw = cg.get_coin_ohlc_by_id(id=coin_id, vs_currency='usd', days=days)
    df = pd.DataFrame(raw, columns=['timestamp', 'open', 'high', 'low', 'close'])
    df['timestamp'] = pd.to_datetime(df['timestamp'], unit='ms')
    df.set_index('timestamp', inplace=True)
    df['coin'] = coin_id
    print(f'  ✅ {len(df)} rows | {df.index[0].date()} → {df.index[-1].date()}')
    return df

dfs = {}
for coin in config.COINS:
    dfs[coin] = fetch_ohlc(coin, days=365)
    time.sleep(1.2)

for coin, df in dfs.items():
    path = f'{config.RAW_DATA}/{coin}_ohlc_1y.csv'
    df.to_csv(path)
    print(f'💾 Saved: {path}')

print('\nBTC sample:')
display(dfs['bitcoin'].tail())

Fetching 365 days of OHLC for bitcoin...
  ✅ 92 rows | 2025-04-18 → 2026-04-17
Fetching 365 days of OHLC for ethereum...
  ✅ 92 rows | 2025-04-18 → 2026-04-17
Fetching 365 days of OHLC for solana...
  ✅ 92 rows | 2025-04-18 → 2026-04-17
💾 Saved: /content/drive/MyDrive/crypto_quant_project/data/raw/bitcoin_ohlc_1y.csv
💾 Saved: /content/drive/MyDrive/crypto_quant_project/data/raw/ethereum_ohlc_1y.csv
💾 Saved: /content/drive/MyDrive/crypto_quant_project/data/raw/solana_ohlc_1y.csv

BTC sample:


,open,high,low,close,coin
timestamp,,,,,
2026-04-01,66328.0,68286.0,65112.0,68232.0,bitcoin
2026-04-05,68231.0,69136.0,65819.0,67304.0,bitcoin
2026-04-09,67306.0,72698.0,66634.0,71117.0,bitcoin
2026-04-13,71004.0,73721.0,70522.0,70757.0,bitcoin
2026-04-17,70651.0,75829.0,70627.0,75149.0,bitcoin


## 2. Market Volume and Capitalization Data

In [ ]:
def fetch_market_chart(coin_id: str, days: int = 365) -> pd.DataFrame:
    """Fetches historical price, volume, and market capitalization data."""
    print(f'Fetching market chart for {coin_id}...')
    data = cg.get_coin_market_chart_by_id(id=coin_id, vs_currency='usd', days=days)

    prices     = pd.DataFrame(data['prices'],      columns=['timestamp', 'price'])
    volumes    = pd.DataFrame(data['total_volumes'], columns=['timestamp', 'volume'])
    market_cap = pd.DataFrame(data['market_caps'],  columns=['timestamp', 'market_cap'])

    df = prices.merge(volumes, on='timestamp').merge(market_cap, on='timestamp')
    df['timestamp'] = pd.to_datetime(df['timestamp'], unit='ms')
    df.set_index('timestamp', inplace=True)

    print(f'  ✅ {len(df)} rows')
    return df

market_dfs = {}
for coin in config.COINS:
    market_dfs[coin] = fetch_market_chart(coin, days=365)
    time.sleep(1.2)

for coin, df in market_dfs.items():
    path = f'{config.RAW_DATA}/{coin}_market_1y.csv'
    df.to_csv(path)
    print(f'💾 Saved: {path}')

display(market_dfs['bitcoin'].tail())

Fetching market chart for bitcoin...
  ✅ 366 rows
Fetching market chart for ethereum...
  ✅ 366 rows
Fetching market chart for solana...
  ✅ 366 rows
💾 Saved: /content/drive/MyDrive/crypto_quant_project/data/raw/bitcoin_market_1y.csv
💾 Saved: /content/drive/MyDrive/crypto_quant_project/data/raw/ethereum_market_1y.csv
💾 Saved: /content/drive/MyDrive/crypto_quant_project/data/raw/solana_market_1y.csv


,price,volume,market_cap
timestamp,,,
2026-04-16 00:00:00,74833.506391,3.359287e+10,1.497468e+12
2026-04-17 00:00:00,75149.193022,4.450075e+10,1.503597e+12
2026-04-18 00:00:00,77128.443349,6.476218e+10,1.543928e+12
2026-04-19 00:00:00,75728.456818,6.629232e+10,1.515913e+12
2026-04-19 11:40:36,75478.615275,4.863547e+10,1.510940e+12


## 3. Live Order Book Ingestion

In [ ]:
import ccxt
import pandas as pd

exchange = ccxt.kraken({'enableRateLimit': True})

def fetch_order_book(symbol: str = 'BTC/USD', depth: int = 10) -> tuple:
    """Fetches live order book depth to calculate liquidity and spread."""
    ob = exchange.fetch_order_book(symbol, limit=depth)

    bids = pd.DataFrame([x[:2] for x in ob['bids']], columns=['price', 'quantity'])
    asks = pd.DataFrame([x[:2] for x in ob['asks']], columns=['price', 'quantity'])

    bids['side'] = 'bid'
    asks['side'] = 'ask'
    return bids, asks

bids, asks = fetch_order_book('BTC/USD', depth=10)

best_bid = bids['price'].max()
best_ask = asks['price'].min()
spread   = best_ask - best_bid
spread_pct = (spread / best_bid) * 100

print('=== LIVE BTC/USD ORDER BOOK ===')
print(f'Best Bid : ${best_bid:,.2f}')
print(f'Best Ask : ${best_ask:,.2f}')
print(f'Spread   : ${spread:.2f} ({spread_pct:.4f}%)\n')
print('Top 5 Bids:')
display(bids.head())
print('\nTop 5 Asks:')
display(asks.head())

=== LIVE BTC/USD ORDER BOOK ===
Best Bid (highest buy order) : $75,489.80
Best Ask (lowest sell order) : $75,489.90
Spread                       : $0.10 (0.0001%)

Top 5 Bids:
     price  quantity side
0  75489.8     5.747  bid
1  75489.7     0.001  bid
2  75489.0     0.525  bid
3  75488.9     0.763  bid
4  75488.1     0.663  bid

Top 5 Asks:
     price  quantity side
0  75489.9     0.005  ask
1  75490.0     0.025  ask
2  75490.1     0.013  ask
3  75491.9     0.018  ask
4  75493.8     0.001  ask


## 4. On-Chain Ethereum Metrics

In [ ]:
rpc_url = 'https://eth.llamarpc.com'
w3 = Web3(Web3.HTTPProvider(rpc_url))

if w3.is_connected():
    print('✅ Connected to Ethereum mainnet')
else:
    print('❌ Connection failed')

latest_block = w3.eth.get_block('latest')
block_num    = latest_block['number']
block_time   = datetime.fromtimestamp(latest_block['timestamp'])
num_txns     = len(latest_block['transactions'])

print(f'\nLatest Block: #{block_num:,}')
print(f'Timestamp   : {block_time}')
print(f'Transactions: {num_txns}')
print(f'Gas Used    : {latest_block["gasUsed"]:,}')
print(f'Gas Limit   : {latest_block["gasLimit"]:,}')
print(f'Utilization : {latest_block["gasUsed"]/latest_block["gasLimit"]*100:.1f}%')

✅ Connected to Ethereum mainnet

Latest Block: #24,913,715
Timestamp   : 2026-04-19 11:44:23
Transactions: 138
Gas Used    : 20,010,230
Gas Limit   : 60,000,000
Utilization : 33.4%


In [ ]:
import time

print('Fetching recent block data...')
block_data = []

safe_block_num = w3.eth.get_block('finalized')['number']

for i in range(20):
    success = False
    while not success:
        try:
            block = w3.eth.get_block(safe_block_num - i)
            block_data.append({
                'block_number': block['number'],
                'timestamp':    datetime.fromtimestamp(block['timestamp']),
                'tx_count':     len(block['transactions']),
                'gas_used':     block['gasUsed'],
                'gas_limit':    block['gasLimit'],
                'utilization':  block['gasUsed'] / block['gasLimit']
            })
            success = True
            time.sleep(1.5)
        except Exception as e:
            err_msg = str(e).lower()
            if '429' in err_msg:
                time.sleep(5)
            elif 'not found' in err_msg or '-32014' in err_msg or 'missing' in err_msg:
                time.sleep(2)
            else:
                raise e

blocks_df = pd.DataFrame(block_data)
path = f'{config.RAW_DATA}/eth_blocks_recent.csv'
blocks_df.to_csv(path, index=False)
print(f'💾 Saved: {path}\n')
display(blocks_df[['block_number', 'tx_count', 'utilization']].head())

Pulling last 20 blocks...
💾 Saved: /content/drive/MyDrive/crypto_quant_project/data/raw/eth_blocks_recent.csv

 block_number  tx_count  utilization
     24913635       203     0.467454
     24913634       228     0.502265
     24913633       229     0.487160
     24913632       385     0.710285
     24913631       185     0.484426
     24913630       154     0.380001
     24913629       209     0.459410
     24913628       246     0.475950
     24913627       294     0.755006
     24913626       301     0.627358
     24913625        79     0.297650
     24913624       247     0.557737
     24913623       229     0.509306
     24913622       389     0.619759
     24913621       214     0.301839
     24913620       236     0.463300
     24913619       226     0.422437
     24913618       247     0.511671
     24913617       255     0.508150
     24913616       210     0.367606


## 5. DeFi Liquidity Data (Uniswap V3)

In [ ]:
UNISWAP_SUBGRAPH = 'https://api.thegraph.com/subgraphs/name/uniswap/uniswap-v3'

query = """
{
  pools(first: 20, orderBy: volumeUSD, orderDirection: desc) {
    id
    token0 { symbol }
    token1 { symbol }
    feeTier
    liquidity
    volumeUSD
    txCount
    totalValueLockedUSD
  }
}
"""

try:
    response = requests.post(UNISWAP_SUBGRAPH, json={'query': query}, timeout=15)

    if response.status_code == 200:
        pools = response.json()['data']['pools']
        pools_df = pd.DataFrame([{
            'pair':       f"{p['token0']['symbol']}/{p['token1']['symbol']}",
            'fee_tier':   int(p['feeTier']) / 10000,
            'volume_usd': float(p['volumeUSD']),
            'tvl_usd':    float(p['totalValueLockedUSD']),
            'tx_count':   int(p['txCount']),
            'pool_id':    p['id']
        } for p in pools])

        path = f'{config.RAW_DATA}/uniswap_pools.csv'
        pools_df.to_csv(path, index=False)
        print(f'💾 Saved: {path}')
        print('\nTop Uniswap V3 Pools by Volume:')
        display(pools_df[['pair', 'fee_tier', 'volume_usd', 'tvl_usd']].head())
    else:
        print('The Graph API returned a non-200 status — falling back to offline sample data.')
except requests.exceptions.RequestException as e:
    print(f'API Connection Error: {e}')
    print('The Graph API unavailable — falling back to offline sample data.')

API Connection Error: HTTPSConnectionPool(host='error.thegraph.com', port=443): Max retries exceeded with url: /apierror.json (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x7e3e19170110>: Failed to resolve 'error.thegraph.com' ([Errno -2] Name or service not known)"))
The Graph API unavailable — using saved sample data if exists


## Summary of Ingested Data

**Persisted Datasets:**
- `*_ohlc_1y.csv`: Daily price action (OHLC).
- `*_market_1y.csv`: Price, volume, and market capitalization.
- `eth_blocks_recent.csv`: On-chain block performance and gas metrics.
- `uniswap_pools.csv`: DeFi liquidity pool statistics.

Data is structured and prepared for feature engineering and quantitative signal generation in subsequent pipelines.